## Analisi Indice di Franosità nel Territorio Umbro 
**Analysis of the Landslide Risk Index in the Umbria Region**
In this notebook, we will calculate the landslide risk index using open data provided by the IFFI inventory.

In questo notebook calcoleremo l'indice di franosità utilizzando gli open data forniti dall'inventario IFFI. (source: https://www.progettoiffi.isprambiente.it/)


### Loading dependencies - Caricamento delle dipendenze
We import the libraries needed for spatial analysis and visualization. We set the paths to \data

Importiamo le librerie necessarie per l'analisi spaziale (`geopandas`) e la visualizzazione.  Stabiliamo i path riferimento a \data

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import mapclassify

BASE_DIR = Path.cwd()
DATA = BASE_DIR / "data"

# GeoDataFrame - .shp limiti Comunali - codici ISTAT col ["PRO_COM"]
# Administrative Boundaries .shp - ISTAT ids listed under col ["PRO_COM"]
gdf_umbria = gpd.read_file(DATA / "umbria.shp").to_crs("EPSG:32632")
gdf_comuni = gpd.read_file(DATA / "comuni_umbria.shp").to_crs("EPSG:32632")

### IFFI Landslide Database import - Importiamo il database frane IFFI
The database was previously converted from .gpckg to .parquet to optimize performance and reduce memory usage.

Il database è stato precedentemente convertito da .gpckg a .parquet per ottimizzare le prestazioni e ridurre il peso sulla memoria.

In [ ]:
# Landslide DB .parquet + data typecasting and downcasting optimization
gdf_frane = (gpd.read_parquet(DATA / "frane.parquet")
             .to_crs("EPSG:32632")
             .astype({
                 "fid": "int32", 
                 "comune": "int32", 
                 "nome_tipo":"category",
                 "nome_distr":"category",
                 "nome_reg":"category",
                 "tipo_movim":"int16",
                 "regione": "int16",
                 "provincia": "int16",
                 "autorita_d": "int32"}))

# check data
gdf_frane.info()

### Area Calculations - Calcolo aree
In order to calculate the landslide risk index, we must first calculate the areas of the individual municipalities and the coverage areas of the individual landslides. We will then group the landslide areas by municipality.

Allo scopo di calcolare l'indice di franosità, dobbiamo prima effettuare il calcolo delle aree dei singoli comuni e l'area di copertura delle singole frane. Raggrupperemo poi le aree delle frane per singolo comune.

In [ ]:
# Aree frane - Landslide Area
gdf_frane["area_frana_m2"] = gdf_frane.geometry.area
gdf_frane["area_frana_Km2"] = gdf_frane.geometry.area / 1_000_000 # conversione a km2

# Aree 'Comuni' - Administrative Boundaries Area
gdf_comuni["area_m2"] = gdf_comuni.geometry.area
gdf_comuni["area_Km2"] = gdf_comuni.geometry.area / 1_000_000 # conversione a km2

# Rimuovo colonne m2 () - remove unused cols
gdf_frane = gdf_frane.drop(columns=["area_frana_m2"])
gdf_comuni = gdf_comuni.drop(columns=["area_m2", "COMUNE_A"])
# Check empty cols
print(gdf_comuni.isna().sum())

# Check data
print("---Checking new created cols--")
print(gdf_frane[["area_frana_Km2", "id_frana", "comune"]].head())
print(gdf_comuni[["area_Km2", "PRO_COM"]].head()) # ["PRO_COM"] = ISTAT id


### Area Intersections - Calcolo intersezioni aree
Since some landslides span multiple municipalities, we will divide the landslide areas along administrative boundaries using an overlay (geopandas) and the `“intersection”` method to ensure the landslide index will be calculated accurately.

Alcune frane con la loro estensione ricadono su più comuni, al fine di rendere accurato il calcolo dell'indice, divideremo le aree di frana tagliandole sui confini amministrativi mediante l'uso di un overlay (geopandas) ed il metodo `"intersection"`.

In [ ]:
# Landslide overlay, 'intersection' with Administrative Boundaries.
# "intersect" - divide la frana sulla base del confine comunale
gdf_frane_comuni = gpd.overlay(
    gdf_frane,
    gdf_comuni[["PRO_COM", "geometry"]], # ["PRO_COM"] = ISTAT id
    how="intersection"
)

print(gdf_frane_comuni.head())


### Landslide Area Aggregation - Aggregazione Area Frane
We group the landslide areas by municipality (using the ISTAT id in col "PRO_COM") and sum them to obtain the total area covered by landslides (in km²) for each municipality. We also calculate the total number of landslides for each municipality using the `‘geometry’ 'count'` function. 

Ragrruppiamo le aree di frana nei rispettivi Comuni (tramite codice ISTAT, col "PRO_COM") ed effettuiamo la somma per ottenere un copertura totale delle frane (in Km2) per ogni Comune.
Calcoliamo anche il numero totale di frane per ogni Comune, tramite `'geometry' 'count'`. 

In [ ]:
# Landslide Aggregation - grouped by Municipalities (ISTAT id)
# Aggregazione Frane - raggruppate per Comune - codice ISTAT
agg = (gdf_frane_comuni.groupby("PRO_COM").agg(num_frane=("geometry", "count"), area_frane_tot=("area_frana_Km2", "sum")).reset_index())
# Data Check
print("---Landslide Aggregation, Data Check---")
print(agg.head())

# Merge with Municipalities gdf - merge con il gdf dei Comuni
gdf_comuni_stats = gdf_comuni.merge(agg, on="PRO_COM", how="left") # "PRO_COM" = ISTAT id

### Landslide Index Calculation - Calcolo Indice di Franosità
We calculate the index by dividing the total landslide area by the total area of the affected territory, then multiply by 100 to obtain values in percent. We check the data distribution using `.describe()`

Calcoliamo l'indice dividendo l'area totale di frana / area territorio interessato, moltiplichiamo per 100 per ottenere valori in %. Controlliamo la distribuzione dei dati con `.describe()` 

In [ ]:
# Landslide Density: Landslide Area / Terrain Area * 100 
# Calcolo densità di frana: Area_frane / Area_territorio * 100 = valore %
gdf_comuni_stats["densita_frane"] = (gdf_comuni_stats["area_frane_tot"] / gdf_comuni_stats["area_Km2"]) * 100

# Check data distribution
print(f"DENISTA' FRANE %: {gdf_comuni_stats["densita_frane"].describe()}")

### Creazione Grafici - Choropleth Map - Landslide Index
The data distribution shows a strong skew (many municipalities with a low risk index and few with a very high risk index). For the graph, we will therefore use the `Natural Breaks (Jenks)` classification scheme, which groups the data in a way that minimizes the variance within each class. We will use 5 risk bands (k=5). 

La distribuzione dei dati mostra una forte asimmetria (molti comuni con indice basso e pochi con indice molto alto). Per il grafico utilizzeremo quindi lo schema di classificazione `Natural Breaks (Jenks)` che raggruppa i dati minimizzando la varianza interna alle classi. Utilizzeremo 5 fasce di rischio k=5 

In [ ]:
# Choropleth map
fig, ax = plt.subplots(figsize=(10, 10))

gdf_comuni_stats.plot(
    column="densita_frane",  # Landslide Index
    cmap="GnBu",
    scheme="NaturalBreaks",  # from mapclassify
    k=5,                     # n. fasce di rischio
    linewidth=0.2,
    edgecolor="black",
    legend=True,
    ax=ax,
    legend_kwds={
        'fmt': "{:.1f}%", 
        'title':"% Territorio Colpito",
        'loc': 'center left',           
        'bbox_to_anchor': (1, 0.7),
    }
)

gdf_umbria.boundary.plot(ax=ax, color="black", linewidth=1)

# Styling
ax.set_axis_off()  # Rimuove assi dal grafico
plt.title("Rischio Idrogeologico Umbria: Indice di Franosità", fontsize=14, fontweight='bold')
plt.tight_layout()

### Number of Landslides by Municipality - Numero di Frane per comune
Here is a bar chart showing the number of landslides recorded for each municipality.
To make the chart clearer, we have decided to show only the municipalities at highest risk, where the number of landslides exceeds the average.

Mostriamo ora un grafico a barre con il numero di frane registrate per ogni Comune.
Per ottenere un grafico più pulito, decidiamo di mostrare solo i comuni a rischio maggiore, con numero di frane superiore al dato medio.

In [ ]:
# Checking Landslide Data Distribution
print("---Landslide Events Stats---")
print(gdf_comuni_stats["num_frane"].describe())

# Filtering Landslide events > 348 (mean) - Filtriamo gli eventi sopra la media
df_filtrato = gdf_comuni_stats[gdf_comuni_stats["num_frane"] > 348].sort_values(by="num_frane", ascending=False)

In [ ]:
# Barplot
plt.figure(figsize=(12,10))

ax = sns.barplot(
    x="num_frane",
    y="COMUNE",
    data=df_filtrato,
    palette="GnBu_r" # reversed
)

# Adding Landslide Counter near bars - contatore frane a dx della barra
for container in ax.containers:
    ax.bar_label(container, padding=5, fontsize=10)

# Styling
plt.title("Numero di Frane per Comune", fontsize=14, fontweight='bold')
plt.xlabel("Numero totale di frane", fontsize=12)
plt.ylabel("")
plt.grid(axis='x', linestyle='--', alpha=0.7) # Griglia verticale per leggere meglio i valori
plt.tight_layout()

### Donut Chart - Landslide Types - Tipologie di Frana
Finally, we present a donut chart showing the types of landslides detected; we will exclude *Subsidence* and *Expansion* landslides, which **account for 0.06%** of the total. 
However, the **IFFI dataset** contains **8.7%** of landslides that have not been classified; therefore, they will be **listed as N/A** (as per the official data).

Infine mostriamo un Donut Chart che indica le tipologie di frana rilevate, escluderemo le frane da *Sprofondamento* ed *Espansione* che rappresentano lo **0.06% del totale**. 
il **Dataset IFFI** tuttavia contiene un **8.7%** di frane che non sono state classificate, pertanto **risulteranno come N.D.** (come dai dati ufficiali)

In [ ]:
from matplotlib.gridspec import GridSpec

# Cheking Landslide Types - Tipologie di Frana
count = gdf_frane["nome_tipo"].value_counts()
print(gdf_frane["nome_tipo"].value_counts())

# Rimuovo outlier che falsano il grafico
count = count.drop(['Sprofondamento', 'Espansione'], errors='ignore')

fig = plt.figure(figsize=(10, 10))
gs = GridSpec(2, 1, height_ratios=[4, 1])  # 4/5 grafico, 1/5 legenda

# --- Grafico Donut ---
ax1 = fig.add_subplot(gs[0])
# Explode chart slice
explode = [0,0,0,0,0,0.3]

patches, texts, autotexts = ax1.pie(
    count.values,
    labels=count.index,  # None per nascondere
    autopct='%1.1f%%',
    startangle=260,
    explode=explode,
    colors=plt.cm.GnBu_r(range(0, 255, 35)),
    pctdistance=0.85,
    wedgeprops={"width": 0.3}
)

ax1.set_title("Categorie di Frana", fontsize=14, fontweight='bold')
plt.setp(autotexts, fontweight='semibold', fontsize=10)

# --- Box Legenda ---
ax2 = fig.add_subplot(gs[1])
ax2.axis("off")  # niente assi

ax2.legend(
    patches,
    count.index,
    title="Tipologie",
    loc="center",
    ncol=3,        # suddivide legenda in n colonne
    frameon=False  # niente bordi 
)

plt.tight_layout()
plt.show()

### Conclusions - Conclusioni
The analysis demonstrates the effectiveness of the Python ecosystem for processing complex environmental data. By integrating various libraries, we achieved the following results:

* Using `geopandas`, we performed overlay operations and geometric calculations (Franosity Index) in a vector-based manner, drastically reducing computation times compared to traditional GIS software.
* Converting the final datasets to **.parquet** format (GeoParquet) ensures optimized storage, preserving geometries and reducing disk space by up to 5–10 times compared to traditional Shapefiles.
* The use of `mapclassify` and `matplotlib` allowed us to transform raw data into an immediate **Choropleth Map**, using the *Natural Breaks* scheme to reflect the actual distribution of risk across the territory.

This approach provides a solid foundation for the development of dynamic and easily scalable hydrogeological instability monitoring systems.

**ITALIAN:**<br>
L'analisi dimostra l'efficacia dell'ecosistema Python per l'elaborazione di dati ambientali complessi. Attraverso l'integrazione di diverse librerie, abbiamo ottenuto i seguenti risultati:

* Grazie a `geopandas`, abbiamo eseguito operazioni di *overlay* e calcoli geometrici (Indice di Franosità) in modo vettoriale, riducendo drasticamente i tempi di calcolo rispetto ai software GIS tradizionali.
* La conversione dei dataset finali in formato **.parquet** (GeoParquet) garantisce un'archiviazione ottimizzata, preservando le geometrie e riducendo lo spazio su disco fino a 5-10 volte rispetto ai classici Shapefile.
* L'uso di `mapclassify` e `matplotlib` ha permesso di trasformare dati grezzi in una **Choropleth Map** immediata, utilizzando lo schema *Natural Breaks* per riflettere la reale distribuzione del rischio sul territorio.

Questo approccio fornisce una base solida per lo sviluppo di sistemi di monitoraggio del dissesto idrogeologico dinamici e facilmente scalabili.
